In [5]:
import sys
sys.path.append("..")

import pandas as pd
from src.alm import (load_balance_sheet, KEYS, nii, equity,
                     delta_eve_curve, delta_nii_curve)

bs = load_balance_sheet()

Reading file: /Users/bimansaha/Projects/bank-rate-risk-model/model/balance_sheet.csv


In [6]:
rbc = {
    "nii_up": 197, "nii_down": -373,        # $M, Table 49
    "eve_up": -2648, "eve_down": 1932,      # $M, Table 49
    "equity": 139151,                       # $M, total equity, Table 36
    "nii_total": 33000,                      # $M, from step 2
}

In [7]:
up = pd.Series(1.0, index=KEYS)
down = pd.Series(-1.0, index=KEYS)
base = nii(bs)

validation = pd.DataFrame({
    "model": [delta_nii_curve(bs, up) / base * 100,
              delta_nii_curve(bs, down) / base * 100,
              delta_eve_curve(bs, up) / equity(bs) * 100,
              delta_eve_curve(bs, down) / equity(bs) * 100],
    "rbc": [rbc["nii_up"] / rbc["nii_total"] * 100,
            rbc["nii_down"] / rbc["nii_total"] * 100,
            rbc["eve_up"] / rbc["equity"] * 100,
            rbc["eve_down"] / rbc["equity"] * 100],
}, index=["NII, +100 bps (% of NII)", "NII, -100 bps (% of NII)",
          "EVE, +100 bps (% of equity)", "EVE, -100 bps (% of equity)"])

validation["same sign?"] = (validation["model"] * validation["rbc"]) > 0
validation.round(2)

,model,rbc,same sign?
"NII, +100 bps (% of NII)",3.17,0.60,True
"NII, -100 bps (% of NII)",-3.17,-1.13,True
"EVE, +100 bps (% of equity)",-15.28,-1.90,True
"EVE, -100 bps (% of equity)",15.28,1.39,True


## Reading the validation table

- All four signs match RBC's Table 49 disclosure: NII rises and EVE falls when rates rise, and the reverse when rates fall.
- My model is far more sensitive than RBC. My EVE moves about 15% of equity, against RBC's 1.9% (+100 bps) and 1.4% (-100 bps), roughly 8 to 11 times larger. My NII moves about 3.2% of NII, against RBC's 0.6% and 1.1%, roughly 3 to 5 times larger.
- RBC's NII is asymmetric (+$197M for +100 bps, -$373M for -100 bps). My model is linear, so its responses are symmetric.
- RBC attributes the change in its EVE sensitivity to net growth in fixed-rate assets, including growth in book capital (Table 49 commentary). My model has the same mechanism, since equity funds assets that carry duration. I could not verify whether hedging explains the rest of the size gap, so that remains a hypothesis.
- RBC's NII percentages use total net interest income from Table 7 ($33,000M), which may include activity outside what Table 49 covers, so they are approximate. My betas, durations, and spreads are assumptions chosen for plausibility, not estimated from data.

In [8]:
print(validation.round(2).to_string())

                             model   rbc  same sign?
NII, +100 bps (% of NII)      3.17  0.60        True
NII, -100 bps (% of NII)     -3.17 -1.13        True
EVE, +100 bps (% of equity) -15.28 -1.90        True
EVE, -100 bps (% of equity)  15.28  1.39        True
